# Bench `apply_changes` sur le corpus réel — PDF + Word

Teste le pattern `extract → modify → rebuild` sur plusieurs fichiers du corpus :
- 5 PDFs représentatifs de `data/` (1 contrat MRH, 1 rapport SFCR, 1 export Word, 1 paper, 1 export Ghostscript)
- 2 docx de `tests/fixtures/`

**Modification appliquée** : pour chaque fichier, on préfixe la 1ère ligne/paragraphe avec `[MODIFIED] ` et on uppercase la 2ème. Pas un cas d'usage métier (ce serait une vraie traduction en prod), juste une preuve que `apply_changes` tourne sans casser sur des fichiers variés.

**Tous les outputs reconstruits sont dans** `notebooks/_outputs/pdf_translation/` et `notebooks/_outputs/word_translation/` pour inspection visuelle dans Adobe Reader / Word.

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import time
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from docpipeline.parsing.pdf.parse_pdf import parse_pdf, apply_changes as apply_pdf
from docpipeline.parsing.word.parse_word import parse_word, apply_changes as apply_docx

REPO = Path('..').resolve()
OUT_PDF  = Path('_outputs/pdf_translation')
OUT_DOCX = Path('_outputs/word_translation')
OUT_PDF.mkdir(parents=True, exist_ok=True)
OUT_DOCX.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_colwidth', 100)

# Sélection de fichiers représentatifs (PDFs courts pour rester < 1 min)
PDF_CASES = [
    REPO / 'data/CG contrats MRH/CG Habitation_MMA_410s.pdf',
    REPO / 'data/annuel_reports/ABEILLE ASSURANCES/09 SFCR 2022 - 10 Abeille Vie.pdf',
    REPO / 'data/insurance/agcs euroapi.pdf',
    REPO / 'data/paper/1706.03762v7.pdf',
    REPO / 'data/cmo/CMO-April-2024.pdf',
]

DOCX_CASES = [
    REPO / 'tests/fixtures/contrat_assurance.docx',
    REPO / 'tests/fixtures/notice_garanties_converted.docx',
]

print(f'PDFs à tester  : {len(PDF_CASES)}')
print(f'DOCX à tester  : {len(DOCX_CASES)}')

PDFs à tester  : 5
DOCX à tester  : 2


In [3]:
# Bench PDF
rows = []
for pdf in PDF_CASES:
    if not pdf.exists():
        rows.append({'file': pdf.name, 'status': 'MISSING'})
        continue
    t0 = time.perf_counter()
    try:
        result = parse_pdf(pdf)
        ld = result['line_df']
        # Modifier 2 premieres lignes : prefixe [MODIFIED] sur la 1ere, uppercase la 2eme
        changes = {}
        if len(ld) >= 1 and ld.iloc[0]['text'].strip():
            changes[ld.iloc[0]['span_id']] = '[MODIFIED] ' + ld.iloc[0]['text']
        if len(ld) >= 2 and ld.iloc[1]['text'].strip():
            changes[ld.iloc[1]['span_id']] = ld.iloc[1]['text'].upper()

        out = OUT_PDF / f'{pdf.stem}_modified_js.pdf'
        res = apply_pdf(pdf, changes, out)

        rows.append({
            'file':           pdf.name,
            'corpus':         pdf.relative_to(REPO / 'data').parts[0] if (REPO / 'data') in pdf.parents else '',
            'n_pages':        result['doc_summary']['n_pages'],
            'source_tool':    result['doc_summary']['source_tool'],
            'n_lines':        len(ld),
            'modifs':         len(changes),
            'spans_replaced': res['spans_replaced'],
            'spans_skipped':  res['spans_skipped'],
            'output_kb':      round(out.stat().st_size / 1024, 1),
            'time_s':         round(time.perf_counter() - t0, 2),
            'status':         'OK',
        })
    except Exception as e:
        rows.append({'file': pdf.name, 'status': f'FAIL: {type(e).__name__}: {e}', 'time_s': round(time.perf_counter() - t0, 2)})

display(Markdown('### Bench PDF — corpus client'))
display(pd.DataFrame(rows))

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


### Bench PDF — corpus client

,file,corpus,n_pages,source_tool,n_lines,modifs,spans_replaced,spans_skipped,output_kb,time_s,status
0,CG Habitation_MMA_410s.pdf,CG contrats MRH,80,Adobe InDesign,4001,2,2,0,458.1,141.36,OK
1,09 SFCR 2022 - 10 Abeille Vie.pdf,annuel_reports,52,Unknown,2292,2,2,0,592.6,47.18,OK
2,agcs euroapi.pdf,insurance,77,Microsoft Word,3205,2,2,0,1587.9,134.88,OK
3,1706.03762v7.pdf,paper,15,Unknown,1048,2,2,0,2060.0,24.02,OK
4,CMO-April-2024.pdf,cmo,52,Ghostscript,6032,2,2,0,6766.4,106.80,OK


In [4]:
# Bench DOCX
rows_docx = []
for docx in DOCX_CASES:
    if not docx.exists():
        rows_docx.append({'file': docx.name, 'status': 'MISSING'})
        continue
    t0 = time.perf_counter()
    try:
        result = parse_word(docx)
        sd = result['span_df']
        # Modifier 2 premiers spans : prefixe [MODIFIED] + uppercase
        changes = {}
        if len(sd) >= 1 and sd.iloc[0]['text'].strip():
            changes[sd.iloc[0]['span_id']] = '[MODIFIED] ' + sd.iloc[0]['text']
        if len(sd) >= 2 and sd.iloc[1]['text'].strip():
            changes[sd.iloc[1]['span_id']] = sd.iloc[1]['text'].upper()

        out = OUT_DOCX / f'{docx.stem}_modified_js.docx'
        apply_docx(docx, changes, out)

        rows_docx.append({
            'file':         docx.name,
            'n_paragraphs': result['doc_summary']['n_paragraphs'],
            'n_spans':      result['doc_summary']['n_spans'],
            'source_tool':  result['doc_summary']['source_tool'],
            'modifs':       len(changes),
            'output_kb':    round(out.stat().st_size / 1024, 1),
            'time_s':       round(time.perf_counter() - t0, 2),
            'status':       'OK',
        })
    except Exception as e:
        rows_docx.append({'file': docx.name, 'status': f'FAIL: {type(e).__name__}: {e}', 'time_s': round(time.perf_counter() - t0, 2)})

display(Markdown('### Bench DOCX — fixtures'))
display(pd.DataFrame(rows_docx))

display(Markdown('### Outputs générés'))
all_outs = sorted(list(OUT_PDF.glob('*_modified_js.pdf')) + list(OUT_DOCX.glob('*_modified_js.docx')))
outs_df = pd.DataFrame([
    {'output': str(p), 'size_kb': round(p.stat().st_size / 1024, 1)}
    for p in all_outs
])
display(outs_df)

### Bench DOCX — fixtures

,file,n_paragraphs,n_spans,source_tool,modifs,output_kb,time_s,status
0,contrat_assurance.docx,10,11,Microsoft Word,2,35.2,0.17,OK
1,notice_garanties_converted.docx,7,10,Microsoft Word,2,35.0,0.17,OK


### Outputs générés

,output,size_kb
0,_outputs\pdf_translation\09 SFCR 2022 - 10 Abeille Vie_modified_js.pdf,592.6
1,_outputs\pdf_translation\1706.03762v7_modified_js.pdf,2060.0
2,_outputs\pdf_translation\agcs euroapi_modified_js.pdf,1587.9
3,_outputs\pdf_translation\CG Habitation_MMA_410s_modified_js.pdf,458.1
4,_outputs\pdf_translation\CMO-April-2024_modified_js.pdf,6766.4
5,_outputs\word_translation\contrat_assurance_modified_js.docx,35.2
6,_outputs\word_translation\notice_garanties_converted_modified_js.docx,35.0
